## Running open weight model on reasoning prompts

### Model : DeepSeek R1 Distill Qwen 1.5B

### Installing dependencies

In [ ]:
!pip install -q -U transformers accelerate

### Imports

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

### GPU is required to load model fast

#### colab T4 gpu is enough

In [ ]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Please enable GPU in Colab.")

### Loading model

In [ ]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.float16,
    device_map = "auto"
)

### Inference function

In [ ]:
def run(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

### Reasoning prompts

In [ ]:
prompt1 = """A train takes 2 hours to go from City A to City B.
The return journey takes 120 minutes.
Which journey is faster?
Show your full reasoning step by step"""

prompt2 = """Rohan had 60 marbles.
He gave 20 to his friend, then doubled what he had left, then lost 15.
How many marbles does he have now?"""

prompt3 = """There are three boxes: one has apples, one has oranges, and one has both.
All labels are wrong.
You can pick one fruit from one box only. Which box should you pick from to fix all labels?
Explain your full chain of reasoning before giving a final answer."""

prompt4 = """Aman can clean a room in 3 hours.
Bina can clean the same room in 6 hours.
They work together for 1 hour, then Aman leaves. How long will Bina take to finish the rest?
Show all steps."""

prompt5 = """A woman looked at a photo and said, “I have no brothers or sisters, but this person’s father is my father’s son.”
Who is in the photo?
Show your reasoning."""

### Prompt 1

In [ ]:
print(run(prompt1))

### Prompt2

In [ ]:
print(run(prompt2))

### Prompt 3

In [ ]:
print(run(prompt3))

### Prompt 4

In [ ]:
print(run(prompt4))

### Prompt 5

In [ ]:
print(run(prompt5))